# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [3]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5051 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [ ]:
import functools

def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Sprawdź argumenty pozycyjne
        for arg in args:
            if isinstance(arg, list):
                print(f"Lista zawiera {len(arg)} elementów.")
        # Sprawdź argumenty nazwane
        for key, value in kwargs.items():
            if isinstance(value, list):
                print(f"Lista '{key}' zawiera {len(value)} elementów.")
        return func(*args, **kwargs)
    return wrapper


# Test:
@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie: {name}")

process_data([1, 2, 3, 4, 5], "moje dane")
process_data([10, 20], name="krótka lista")
process_data("tekst", "brak listy")

### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [ ]:
import functools
from datetime import datetime
import time

def logger(filename):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start_time = time.perf_counter()
            result = func(*args, **kwargs)
            end_time = time.perf_counter()
            duration = end_time - start_time
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log_entry = f"[{timestamp}] Funkcja: {func.__name__} | Czas wykonania: {duration:.4f} s\n"
            with open(filename, "a", encoding="utf-8") as log_file:
                log_file.write(log_entry)
            print(log_entry.strip())
            return result
        return wrapper
    return decorator


# Test:
@logger("aplikacja.log")
def oblicz_sume(a, b):
    time.sleep(0.1)
    return a + b

@logger("aplikacja.log")
def pobierz_dane():
    time.sleep(0.2)
    return [1, 2, 3]

wynik = oblicz_sume(3, 7)
print(f"Wynik: {wynik}")
dane = pobierz_dane()
print(f"Dane: {dane}")

--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [6]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [ ]:
class AccessLogger:
    def __set_name__(self, owner, name):
        self.public_name = name
        self.private_name = f"_logged_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = instance.__dict__.get(self.private_name, None)
        print(f"[ODCZYT] Atrybut '{self.public_name}' = {value!r}")
        return value

    def __set__(self, instance, value):
        stara_wartosc = instance.__dict__.get(self.private_name, None)
        instance.__dict__[self.private_name] = value
        print(f"[ZAPIS]  Atrybut '{self.public_name}': {stara_wartosc!r} -> {value!r}")


class Uzytkownik:
    imie = AccessLogger()
    wiek = AccessLogger()

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek


# Test:
print("--- Tworzenie użytkownika ---")
u = Uzytkownik("Anna", 25)

print("\n--- Odczyt atrybutów ---")
print(u.imie)
print(u.wiek)

print("\n--- Zmiana wartości ---")
u.imie = "Maria"
u.wiek = 30

print("\n--- Ponowny odczyt ---")
print(u.imie)
print(u.wiek)

--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [ ]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [ ]:
def collatz_generator(n):
    """Generator ciągu Collatza startujący od n, kończący na 1."""
    if n < 1:
        raise ValueError("Liczba startowa musi być dodatnią liczbą naturalną.")
    while n != 1:
        yield n
        if n % 2 == 0:
            n = n // 2
        else:
            n = 3 * n + 1
    yield 1  # Zwróć końcową wartość 1


# Test:
print("Ciąg Collatza dla n=10:")
for value in collatz_generator(10):
    print(value, end=" ")
print()

print("\nCiąg Collatza dla n=27:")
sekwencja = list(collatz_generator(27))
print(sekwencja)
print(f"Długość ciągu: {len(sekwencja)} kroków")

---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [ ]:
import functools

current_user = {"username": "admin", "role": "superuser"}

def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            user_role = current_user.get("role")
            if user_role != role:
                raise PermissionError(
                    f"Brak uprawnień: wymagana rola '{role}', "
                    f"aktualny użytkownik ma rolę '{user_role}'."
                )
            return func(*args, **kwargs)
        return wrapper
    return decorator


# Test:
@require_role("superuser")
def usun_uzytkownika(user_id):
    print(f"Usunięto użytkownika o ID: {user_id}")

@require_role("admin")
def panel_administracyjny():
    print("Dostęp do panelu administracyjnego.")


print(f"Zalogowany: {current_user['username']} (rola: {current_user['role']})\n")

# Powinno zadziałać – rola 'superuser' pasuje
try:
    usun_uzytkownika(42)
except PermissionError as e:
    print(f"Błąd: {e}")

# Powinno rzucić PermissionError – rola 'admin' != 'superuser'
try:
    panel_administracyjny()
except PermissionError as e:
    print(f"Błąd: {e}")

### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [ ]:
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.public_name = name
        self.private_name = f"_typed_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.private_name)

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"Atrybut '{self.public_name}' musi być typu "
                f"{self.expected_type.__name__}, "
                f"otrzymano {type(value).__name__}."
            )
        instance.__dict__[self.private_name] = value


# Przykładowe użycie:
class Produkt:
    nazwa = Typed(str)
    cena = Typed(float)
    ilosc = Typed(int)

    def __init__(self, nazwa, cena, ilosc):
        self.nazwa = nazwa
        self.cena = cena
        self.ilosc = ilosc

    def __repr__(self):
        return f"Produkt({self.nazwa!r}, {self.cena}, {self.ilosc})"


# Test:
print("--- Poprawne dane ---")
p = Produkt("Laptop", 3499.99, 5)
print(p)

print("\n--- Błędny typ dla 'cena' ---")
try:
    p.cena = "drogi"
except TypeError as e:
    print(f"Błąd: {e}")

print("\n--- Błędny typ dla 'ilosc' ---")
try:
    Produkt("Mysz", 49.99, "pięć")
except TypeError as e:
    print(f"Błąd: {e}")

### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [ ]:
def prime_generator():
    """Nieskończony generator liczb pierwszych (inkrementalne sito Eratostenesa)."""
    composites = {}  # liczba złożona -> jej najmniejszy dzielnik pierwszy
    candidate = 2

    while True:
        if candidate not in composites:
            yield candidate
            composites[candidate * candidate] = candidate
        else:
            prime = composites.pop(candidate)
            multiple = candidate + prime
            while multiple in composites:
                multiple += prime
            composites[multiple] = prime
        candidate += 1


# Wyrażenie generatorowe – liczby pierwsze kończące się na 7
primes_ending_7 = (p for p in prime_generator() if p % 10 == 7)


# Test:
import itertools

print("Pierwsze 15 liczb pierwszych kończących się cyfrą 7:")
wynik = list(itertools.islice(primes_ending_7, 15))
print(wynik)

print("\nPierwsze 20 liczb pierwszych (ogólnie):")
print(list(itertools.islice(prime_generator(), 20)))